# Pregunta 2 — Hi ha relació entre el nombre de vots i la recaptació?
**Projecte:** Anàlisi d'un conjunt de dades (0378 – Seguretat i alta disponibilitat, Tema 1)
**Dataset:** `IMDB-Movie-Data.csv` — 1.000 pel·lícules (2006–2016)

Aquest notebook treballa **una única pregunta de recerca**, amb el procediment complet: preparació de les dades, mètode d'anàlisi, resultat i interpretació.

## 1. Preparació de les dades

**Procediment:**
- Carreguem el CSV en un `DataFrame` de Pandas.
- Renombrem columnes amb espais/parèntesis (`runtime`, `revenue`).
- Convertim `Revenue = 0` a `NaN` (no és una recaptació real, és un valor desconegut).
- Separem `genre` i `actors` en llistes (cada pel·lícula en pot tenir diversos).
- Creem un subconjunt `df_fiable` amb `votes >= 1000`, perquè les pel·lícules amb pocs vots tenen una nota poc representativa (sobretot les de 2016).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("IMDB-Movie-Data.csv", encoding="utf-8")
df = df.rename(columns={"Runtime (Minutes)": "runtime", "Revenue (Millions)": "revenue"})
df.columns = [c.lower() for c in df.columns]

# Revenue = 0 és, amb tota probabilitat, un valor desconegut, no una recaptació real
df.loc[df["revenue"] == 0, "revenue"] = np.nan

# Gèneres i actors com a llista (eren text separat per comes)
df["genres"] = df["genre"].str.split(",").apply(lambda l: [g.strip() for g in l])
df["actors_list"] = df["actors"].str.split(",").apply(lambda l: [a.strip() for a in l])

# Subconjunt amb vots suficients per considerar la nota fiable
df_fiable = df[df["votes"] >= 1000].copy()

print(f"Total de pel·lícules: {len(df)}")
print(f"Amb >= 1000 vots (notes fiables): {len(df_fiable)}")
df.head(3)

## 2. Pregunta de recerca

**Hi ha relació entre el nombre de vots a IMDb (`Votes`) i la recaptació (`Revenue`)?**

**Mètode:** descartem les pel·lícules sense `revenue`. Calculem la correlació de Pearson (relació lineal) i la de Spearman (relació d'ordre, més robusta als valors extrems). Fem servir Spearman com a referència principal perquè ambdues variables tenen una distribució molt asimètrica (unes poques pel·lícules concentren la majoria de vots i recaptació).

In [ ]:
r = df.dropna(subset=["revenue"]).copy()
print(f"Pel·lícules amb recaptació disponible: {len(r)} de {len(df)}")

pearson = r["votes"].corr(r["revenue"])
spearman = r["votes"].corr(r["revenue"], method="spearman")
print(f"Correlació de Pearson:  {pearson:.3f}")
print(f"Correlació de Spearman: {spearman:.3f}")
print(f"Asimetria (skew) de votes: {r['votes'].skew():.2f} | de revenue: {r['revenue'].skew():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.scatterplot(data=r, x="votes", y="revenue", alpha=0.45, s=20, ax=ax)
ax.set_xscale("log")
ax.set_title("Vots (escala logarítmica) vs recaptació (M)")
ax.set_xlabel("Vots (escala log)"); ax.set_ylabel("Recaptació (milions)")
plt.tight_layout(); plt.show()

## 3. Resultats

- Correlació de Pearson **0,64** i de Spearman **0,70** (n = 871): és la correlació més forta de totes les calculades en aquest projecte.
- La relació és clarament positiva: les pel·lícules amb més vots (més populars/comentades a IMDb) tendeixen a ser les que més recapten.
- Al gràfic de dispersió (amb l'eix de vots en escala logarítmica perquè la distribució és molt asimètrica) es veu una tendència ascendent clara, encara que amb bastant dispersió: hi ha pel·lícules amb molts vots i recaptació moderada, i viceversa.


## 4. Interpretació i limitacions

✏️ **Omple-ho tu:** és lògic que ambdues variables estiguin relacionades, perquè reflecteixen popularitat de formes diferents. Pensa si la relació podria ser deguda a una tercera variable (pressupost de màrqueting, nombre de sales) que el dataset no recull. Recorda que 129 pel·lícules no tenen `revenue`.